importamos las librerias, le indicamos a python donde encontrar el archivo utils.py y cargamos el dataset que se limpio en 03_limpieza

In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Permitir que el cuaderno lea las funciones de la carpeta src
sys.path.append(os.path.abspath('../src'))
from utils import limpiar_generos, asignar_temporada

# Cargar los datos limpios y auditados 
ruta_limpia = '../data/processed/movies_unificado.csv'
df_transformado = pd.read_csv(ruta_limpia)

print(f"Dataset cargado con {df_transformado.shape[0]} filas y {df_transformado.shape[1]} columnas.")

Dataset cargado con 3229 filas y 24 columnas.


Tomamos la columna de texto release_date la convertimos a un formato de tiempo real (datatime) y extraemos atributos clave: el mes y el dia de la semana. luego, aplicamos la funcion modular para crear la columna "Temporada". esto transforma a un dato crudo en variables de altisimo valor para el modelo predictivo

In [2]:
# Convertir texto a formato datetime de Pandas
df_transformado['release_date'] = pd.to_datetime(df_transformado['release_date'])

# Extracción rápida de características
df_transformado['mes_estreno'] = df_transformado['release_date'].dt.month
df_transformado['dia_semana_estreno'] = df_transformado['release_date'].dt.dayofweek # 0=Lunes, 6=Domingo

# Aplicar la función modular de temporadas
df_transformado['temporada'] = df_transformado['mes_estreno'].apply(asignar_temporada)

print("Atributos temporales (Mes, Día, Temporada) extraídos exitosamente.")

Atributos temporales (Mes, Día, Temporada) extraídos exitosamente.


Primero, se aplana el json de géneros. Luego, utilizando el análisis de 01_eda, se crean variables numericas binarias

**Los modelos de Machine Learning no entienden palabras, solo números, por lo que este One-Hot Encoding manual es indispensable.**

In [3]:
# Aplanar la estructura JSON usando la función optimizada
df_transformado['generos_lista'] = df_transformado['genres'].apply(limpiar_generos)

# Encoding manual para el Top 3 de géneros identificados en la Fase 1
df_transformado['es_drama'] = df_transformado['generos_lista'].apply(lambda x: 1 if 'Drama' in x else 0)
df_transformado['es_comedia'] = df_transformado['generos_lista'].apply(lambda x: 1 if 'Comedy' in x else 0)
df_transformado['es_thriller'] = df_transformado['generos_lista'].apply(lambda x: 1 if 'Thriller' in x else 0)

print("Géneros aplanados y codificados numéricamente.")

Géneros aplanados y codificados numéricamente.


Calculamos el Retorno de Inversión ($ROI = \frac{Revenue - Budget}{Budget}$) operando columnas completas a la vez, lo cual es exponencialmente más rápido que usar un bucle for. Finalmente, reducimos el consumo de memoria RAM cambiando los tipos de datos de texto repetitivo a category.

In [4]:
# Cálculo Vectorizado / Broadcasting para el ROI
df_transformado['roi'] = (df_transformado['revenue'] - df_transformado['budget']) / df_transformado['budget']

# Optimización de Memoria (Chunking conceptual de tipos de datos)
# Convertir variables de texto con pocas opciones únicas a tipo 'category'
columnas_a_optimizar = ['original_language', 'status', 'temporada']
for col in columnas_a_optimizar:
    df_transformado[col] = df_transformado[col].astype('category')

# Mostrar el ahorro de memoria y los nuevos datos
print("Memoria optimizada y ROI calculado.")
df_transformado[['title', 'release_date', 'temporada', 'roi', 'es_drama']].head()

Memoria optimizada y ROI calculado.


KeyError: "['title'] not in index"

In [ ]:
# Guardado del Dataset Final

ruta_final = '../data/processed/movies_listo_para_modelo.csv'
df_transformado.to_csv(ruta_final, index=False)

print(f"Dataset final guardado en: {ruta_final}")

Dataset final guardado en: ../data/processed/movies_listo_para_modelo.csv
